# Selective Recall — Go/No-Go Benchmark

**The decisive question.** Does routing attention to only a fraction of tokens, over
a bounded memory, actually save wall-clock time and peak memory at long context,
versus dense full attention? If yes, Selective Recall has a real efficiency story
worth a strong paper. If no, no amount of writing will fix it.

This benchmark isolates the **attention pathway**, because that is the only part the
gate changes. It compares, as context length `L` grows:

- **DENSE** — standard causal attention over the full context (uses the fused
  FlashAttention kernel on GPU; memory and compute that a real Transformer pays).
- **SPARSE** — the Selective Recall pathway: only `rho*L` queries attend, over a
  **bounded** memory of `M` entries, via a real gather → attend → scatter.

**Read this before trusting any number:**
1. **Use a GPU.** `Runtime → Change runtime type → GPU`. On CPU the fused kernel is
   not used and memory profiling is unavailable, so CPU results are misleading and
   overstate the sparse win. The decision must be made on GPU.
2. This measures the attention pathway only. The full model also runs a state-space
   layer over all `L` tokens, a cost linear in `L`, not included here.
3. The bounded memory is an approximation of full context. Whether `M` entries keep
   accuracy is a **separate** question (the recall experiments). This measures cost.

**What a "GO" looks like:** on GPU, the sparse curve is clearly below dense in both
latency and peak memory at long context (say `L >= 8K`), with the gap widening as `L`
grows. **What a "NO-GO" looks like:** the dense FlashAttention kernel stays at or
below sparse across the practical range, or the memory advantage is small.


## 1. Setup and GPU check

In [ ]:
import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Enable one via Runtime -> Change runtime type -> GPU.")
    print("CPU results are misleading for this benchmark; see the note above.")

## 2. Benchmark code
The two attention pathways, the gather/scatter, and the timing and memory probes.

In [ ]:
import time
import statistics

import torch
import torch.nn as nn
import torch.nn.functional as F


# =====================================================================
# Attention pathways
# =====================================================================
class DenseAttention(nn.Module):
    """Standard causal multi-head attention over the full context."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, L, self.h, self.dk).transpose(1, 2)
        k = k.view(B, L, self.h, self.dk).transpose(1, 2)
        v = v.view(B, L, self.h, self.dk).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(out.transpose(1, 2).reshape(B, L, D))


class SparseBoundedAttention(nn.Module):
    """Selective Recall attention pathway.

    A fraction rho of query positions are selected. Those queries are gathered,
    attend over a bounded memory of M entries, and the result is scattered back.
    The memory is the most recent M tokens (a stand-in for the compressed memory;
    its size, not its contents, drives cost).
    """
    def __init__(self, d_model, n_heads, memory_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.M = memory_size
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, sel_idx):
        # x: (B, L, D); sel_idx: (B, k) indices of the selected query positions
        B, L, D = x.shape
        k_sel = sel_idx.shape[1]

        # bounded memory: the last M tokens -> keys, values
        mem = x[:, -self.M:, :]                                  # (B, M, D)
        kv = self.kv_proj(mem)
        kmem, vmem = kv.chunk(2, dim=-1)
        kmem = kmem.view(B, self.M, self.h, self.dk).transpose(1, 2)
        vmem = vmem.view(B, self.M, self.h, self.dk).transpose(1, 2)

        # gather the selected queries: (B, k, D)
        gather_idx = sel_idx.unsqueeze(-1).expand(B, k_sel, D)
        q_sel = torch.gather(x, 1, gather_idx)
        q = self.q_proj(q_sel).view(B, k_sel, self.h, self.dk).transpose(1, 2)

        # selected queries attend over the bounded memory
        out = F.scaled_dot_product_attention(q, kmem, vmem)      # (B, h, k, dk)
        out = self.proj(out.transpose(1, 2).reshape(B, k_sel, D))

        # scatter back into a full-length output
        y = x.new_zeros(B, L, D)
        y.scatter_(1, gather_idx, out)
        return y


def make_selection(B, L, k, device):
    """Random distinct selected positions per row: (B, k) long."""
    idx = torch.stack([torch.randperm(L, device=device)[:k] for _ in range(B)])
    return idx.sort(dim=1).values


# =====================================================================
# Measurement
# =====================================================================
def _sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def measure_latency(fn, iters=10, warmup=3):
    """Median forward latency in milliseconds."""
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(iters):
        _sync_dev()
        t0 = time.perf_counter()
        fn()
        _sync_dev()
        times.append((time.perf_counter() - t0) * 1000.0)
    return statistics.median(times)


# module-level device for sync (set in run)
_DEVICE = torch.device("cpu")
def _sync_dev():
    if _DEVICE.type == "cuda":
        torch.cuda.synchronize()


def measure_peak_memory(fn):
    """Peak allocated memory in MB during fn (CUDA only)."""
    if _DEVICE.type != "cuda":
        return None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    fn()
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 2)


# =====================================================================
# Benchmark driver
# =====================================================================
def run(L_values, rho_values, M, d_model, n_heads, batch, device, dtype,
        iters, warmup):
    global _DEVICE
    _DEVICE = device
    dense = DenseAttention(d_model, n_heads).to(device=device, dtype=dtype).eval()

    results = []
    print(f"\ndevice={device.type}  dtype={dtype}  d_model={d_model}  "
          f"heads={n_heads}  batch={batch}  memory M={M}")
    print(f"iters={iters} (warmup {warmup})\n")
    header = ["L", "dense_ms"] + [f"sparse_ms(r={r})" for r in rho_values]
    if device.type == "cuda":
        header += ["dense_MB"] + [f"sparse_MB(r={r})" for r in rho_values]
    print("  ".join(f"{h:>16s}" for h in header))

    with torch.no_grad():
        for L in L_values:
            x = torch.randn(batch, L, d_model, device=device, dtype=dtype)

            # dense
            try:
                dense_ms = measure_latency(lambda: dense(x), iters, warmup)
                dense_mb = measure_peak_memory(lambda: dense(x))
                dense_ok = True
            except RuntimeError as e:            # e.g. OOM at large L
                dense_ms, dense_mb, dense_ok = float("nan"), None, False
                torch.cuda.empty_cache() if device.type == "cuda" else None

            row = {"L": L, "dense_ms": dense_ms, "dense_MB": dense_mb}
            sparse_ms_list, sparse_mb_list = [], []
            for r in rho_values:
                k = max(1, int(r * L))
                k = min(k, L)
                sparse = SparseBoundedAttention(
                    d_model, n_heads, min(M, L)).to(device=device, dtype=dtype).eval()
                sel = make_selection(batch, L, k, device)
                sms = measure_latency(lambda: sparse(x, sel), iters, warmup)
                smb = measure_peak_memory(lambda: sparse(x, sel))
                sparse_ms_list.append(sms)
                sparse_mb_list.append(smb)
            row["sparse_ms"] = sparse_ms_list
            row["sparse_MB"] = sparse_mb_list
            results.append(row)

            cells = [f"{L:>16d}", f"{dense_ms:>16.2f}"]
            cells += [f"{s:>16.2f}" for s in sparse_ms_list]
            if device.type == "cuda":
                cells += [f"{(dense_mb or 0):>16.1f}"]
                cells += [f"{(s or 0):>16.1f}" for s in sparse_mb_list]
            print("  ".join(cells))
    return results


def plot(results, rho_values, device_type, path_prefix="bench"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    Ls = [r["L"] for r in results]
    # latency
    plt.figure(figsize=(6.4, 4.6))
    plt.plot(Ls, [r["dense_ms"] for r in results], "o-", color="#4d4d4d",
             label="dense full attention")
    colors = ["#6a51a3", "#2c7fb8", "#d95f0e", "#31a354"]
    for j, rho in enumerate(rho_values):
        plt.plot(Ls, [r["sparse_ms"][j] for r in results], "s-",
                 color=colors[j % len(colors)],
                 label=f"sparse+bounded (rho={rho})")
    plt.xlabel("context length L")
    plt.ylabel("forward latency (ms)")
    plt.title("Attention pathway latency vs context length")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{path_prefix}_latency.png", dpi=140)
    print(f"saved {path_prefix}_latency.png")

    # latency, log-log (to see quadratic vs linear slopes)
    plt.figure(figsize=(6.4, 4.6))
    plt.loglog(Ls, [r["dense_ms"] for r in results], "o-", color="#4d4d4d",
               label="dense full attention")
    for j, rho in enumerate(rho_values):
        plt.loglog(Ls, [r["sparse_ms"][j] for r in results], "s-",
                   color=colors[j % len(colors)], label=f"sparse (rho={rho})")
    plt.xlabel("context length L (log)")
    plt.ylabel("latency ms (log)")
    plt.title("Slopes: dense ~ quadratic, sparse+bounded ~ linear")
    plt.legend()
    plt.grid(alpha=0.3, which="both")
    plt.tight_layout()
    plt.savefig(f"{path_prefix}_latency_loglog.png", dpi=140)
    print(f"saved {path_prefix}_latency_loglog.png")

    if device_type == "cuda" and results[0]["dense_MB"] is not None:
        plt.figure(figsize=(6.4, 4.6))
        plt.plot(Ls, [r["dense_MB"] for r in results], "o-", color="#4d4d4d",
                 label="dense full attention")
        for j, rho in enumerate(rho_values):
            plt.plot(Ls, [r["sparse_MB"][j] for r in results], "s-",
                     color=colors[j % len(colors)], label=f"sparse (rho={rho})")
        plt.xlabel("context length L")
        plt.ylabel("peak memory (MB)")
        plt.title("Attention pathway peak memory vs context length")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{path_prefix}_memory.png", dpi=140)
        print(f"saved {path_prefix}_memory.png")

## 3. Run the benchmark

Defaults sweep `L` up to 32K. If you hit an out-of-memory error, remove the largest
`L` values or reduce `batch`. `rho` is the attending fraction; `M` is the bounded
memory size. `d_model=512`, `8` heads, and `fp16` match a realistic setting.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32

L_values = [1024, 2048, 4096, 8192, 16384, 32768]   # trim if OOM
rho_values = [0.1, 0.2, 0.5]
M = 256           # bounded memory size
d_model, n_heads, batch = 512, 8, 1
iters, warmup = 20, 5

results = run(L_values, rho_values, M, d_model, n_heads, batch,
              device, dtype, iters, warmup)

## 4. Plots
Latency (linear and log-log) and, on GPU, peak memory.

In [ ]:
import matplotlib
%matplotlib inline
plot(results, rho_values, device.type)
from IPython.display import Image, display
for name in ["bench_latency.png", "bench_latency_loglog.png", "bench_memory.png"]:
    try:
        display(Image(name))
    except Exception:
        pass

## 5. Read the result
The crossover summary prints where (if anywhere) sparse beats dense.

In [ ]:
print("Latency crossover (attention pathway):")
for j, r in enumerate(rho_values):
    crossed = None
    for row in results:
        d, s = row["dense_ms"], row["sparse_ms"][j]
        if s == s and d == d and s < d:
            crossed = row["L"]; break
    if crossed is not None:
        print(f"  rho={r}: sparse becomes faster than dense at L >= {crossed}")
    else:
        print(f"  rho={r}: sparse was NOT faster than dense at any tested L")

if device.type == "cuda" and results[0]["dense_MB"] is not None:
    print("\nPeak memory at the largest L:")
    last = results[-1]
    print(f"  L={last['L']}: dense={last['dense_MB']:.0f} MB")
    for j, r in enumerate(rho_values):
        print(f"            sparse(rho={r})={last['sparse_MB'][j]:.0f} MB")

print("""
GO if, on GPU, sparse is clearly below dense in BOTH latency and memory at long
context, with the gap widening as L grows. That is the seed of a strong result:
pair it with an accuracy check (the recall experiments) to show the bounded memory
keeps quality, then scale up and add real baselines (Mamba kernel, a strong hybrid,
Mixture-of-Depths).

NO-GO if the dense FlashAttention kernel stays competitive in the practical range.
That is still a clean, honest workshop result about why a plausible idea does not
pay off. Either way, you now have the answer instead of assuming it.
""")